# Causal ranking feature dataset

BT4222 Group 6 · feature engineering. Run the three `3_Representations` notebooks (3_01, 3_02, 3_03) first.

One shared feature API supports complete logged slates and unlabelled retrieved Top-M candidates.
Historical state always consumes the **complete earlier exposure stream**, even when emitted training
pairs are sampled. Validation/test slates stay complete. This notebook defaults to a chronological smoke
build; set `MAX_TIMESTAMPS=None` for all agreed data once the smoke audit passes.

Run top to bottom. Shared algorithms, validation and file handling live in `src/newsrec/`; this notebook controls settings and shows results. You do not run those Python files separately. Saved artifacts and `REBUILD` work as before.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents,
                            Path('/content/drive/MyDrive/BT4222'))
                  if (p / 'src/newsrec').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('Open this notebook from the project checkout containing src/newsrec/.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = None  # Optional: point to another folder holding the project's saved datasets.
ROOT = Path(DATA_ROOT or os.environ.get('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Data/output root:', ROOT)

from newsrec.features import feature_schema
from newsrec.pipeline import verify_feature_parity
from newsrec.io import feature_run, feature_sample

In [ ]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

## 1. Inspect the model contract

Core groups: content, history, match, repeated exposure, smoothed popularity and observed age.
Report extensions: three-interest matching and supported past-only half-life. Request context is provided
for ablation; slate relative advantage is opt-in (`context_features=true`) and uses the complete original
logged slate before sampling. At inference it must use the actual retrieved candidate set.

IDs, targets, shuffled position, slate length and current click-count summaries are outside the model allowlist.
Missing cosines/half-life use zero plus masks. Half-life compares disjoint completed six-hour windows,
uses Beta(1,1) smoothed CTR, and requires configurable support (100 exposures per window by default).
This specifies a new reproducible estimator; the report's original notebook is absent.

In [ ]:
schema = feature_schema(SETTINGS)
display(pd.DataFrame(schema["features"]))
assert not set(schema["model_columns"]) & set(schema["excluded"])

## 2. Replay and write features

Prequential mode freezes fitted representation/model parameters but updates observed feedback after each
timestamp. Frozen mode consumes earlier splits only and needs `TARGET_SPLIT`. Both modes advance the clock;
age and time-window queries still use the request time. The shared train/validation boundary second is processed
as one batch. `OVERWRITE=True` rebuilds only the selected run directory.

Training sampling is a seeded per-positive draw of up to four exposed negatives without replacement.
The pointwise feature table deduplicates the union of selected pairs; grouped models use the separate
positive-to-negative membership table. All popularity state is calculated before this sampling.

The SQLite user/article history bounds RAM; it requires local disk space. Full-run time, disk size and peak
RAM on MIND-large must be measured: a successful smoke run is not a full-scale performance claim.

In [ ]:
# A notebook run defaults to a small smoke build; the terminal runner defaults to full.
MAX_TIMESTAMPS = (None if os.environ.get('NEWSREC_FULL_FEATURES') == '1'
                  else int(os.environ.get('NEWSREC_SMOKE_TIMESTAMPS', '1000')))
TARGET_SPLIT = None  # Set validation/test with Settings(update_mode='frozen').
OVERWRITE = REBUILD or os.environ.get('NEWSREC_OVERWRITE_FEATURES') == '1'
run_dir = feature_run(ROOT, SETTINGS, target_split=TARGET_SPLIT,
                      max_timestamps=MAX_TIMESTAMPS, rebuild=OVERWRITE)
print(run_dir)
display(json.loads((run_dir / 'run_manifest.json').read_text())['rows_written'])

## 3. Audit coverage, sampling and values

Reports describe all scored logged pairs before training sampling; target tables are stored separately.
Do not load the full MIND feature dataset into pandas at once: select columns or use Arrow/Polars scans.
A smoke starting at training start may contain no validation/test output; absent partitions are expected.

In [ ]:
display(pd.read_csv(run_dir / 'coverage.csv'))
display(pd.read_csv(run_dir / 'feature_quality.csv').head(20))
display(pd.read_csv(run_dir / 'feature_cost.csv'))
display(json.loads((run_dir / 'sampling_summary.json').read_text()))
display(feature_sample(run_dir, split='train', rows=8))

## 4. Verify unlabelled inference parity

The later retrieval notebook supplies `(impression_uid, news_code)` pairs from the eligible catalogue.
No label/position is required. This initial request example uses the same pairs as the logged slate to
verify identical features. For a different Top-M set, only optional candidate-set context may change.

In [ ]:
verify_feature_parity(ROOT, SETTINGS)
print('Logged/retrieved feature parity passed; no target fields needed.')

## Handoff and limitations

Read `feature_schema.json` and `run_manifest.json` before loading any partitions. Use only `model_columns`
and apply train-fitted downstream preprocessing. Each run contains keyed labels, request profile references,
negative groups, sampling summary, coverage and quality audits. A run must have `run_status.json: complete`;
its manifest explicitly identifies smoke/full status and temporal mode. Unshown retrieved items remain unlabelled.
Graph embeddings, K-means, learned contextual encoders, ANN retrieval and model training belong to later work.
Core feature/runtime coverage does not establish ranking improvement or live CTR uplift.